# Matplotlib 之二：散点图（scatter）

---

散点图用来观察**两个连续变量之间的关系**——比如"学习时长"和"考试成绩"是不是正相关、"身高"和"体重"是不是一起变。每个点是一对 `(x, y)`。

## 学习目标

1. 用 `plt.scatter()` 画散点，知道它和折线图的区别
2. 用 `s`（大小）、`c`（颜色）、`alpha`（透明度）、`cmap`（颜色映射）表达第 3、4 维信息
3. 会用 `np.polyfit` 画一条**回归线**看整体趋势
4. 再次巩固中文显示配置（本项目头号坑）


---
## 一、什么时候用散点图？

| 图类型 | 适合回答的问题 | 数据特征 |
|---|---|---|
| 折线图 | 随**时间/顺序**怎么变 | x 是有序的连续量 |
| 散点图 | 两个量**是否相关** | x、y 都是连续量，看重"点的分布形状" |
| 柱状图 | 各**类别**谁多谁少 | x 是离散类别 |

一句话：**想知道"这俩变量有没有关系"，就画散点图**。如果点大致沿一条斜线排，就是线性相关；如果一团乱麻，就基本没关系。


---
## 二、最基础的散点图

`plt.scatter(x, y)` 和 `plot` 很像，但**点之间不连线**，只画一个个标记。

> **原稿问题**：原稿把 `font.sans-serif` 只设成 `['FangSong']` 一个字体。一旦这台电脑没装仿宋，中文立刻变方块，而且没有任何兜底。**正确做法是给一个字体列表**，让 matplotlib 自动挑第一个装了的（见下一节）。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'FangSong', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 模拟 50 个学生的"学习时长"和"成绩"
np.random.seed(0)
study_hours = np.random.uniform(1, 5, 50)          # 1~5 小时
scores = 30 + 15 * study_hours + np.random.normal(0, 8, 50)   # 和时长正相关，带噪声

plt.figure(figsize=(10, 6))
plt.scatter(study_hours, scores)
plt.title("学习时间与成绩的关系")
plt.xlabel("学习时间（小时）")
plt.ylabel("成绩")
plt.show()   # 输出: 右下→左上的点云，大致正相关的趋势


---
## 三、中文显示问题（再讲一次，务必记牢）

散点图只要带中文标题/标签，就躲不开这个坑。原理和折线图完全一样：**matplotlib 默认字体不含中文，显示成方块**。

解决办法还是改 `rcParams`：

```python
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
```

- **Windows** 优先 `SimHei` / `Microsoft YaHei`
- **macOS** 优先 `Arial Unicode MS` / `PingFang SC`
- **Linux** 优先 `WenQuanYi Zen Hei` / `Noto Sans CJK SC`
- 末尾放 `DejaVu Sans` 兜底，保证任何环境都不至于崩溃

`axes.unicode_minus = False` 千万不能漏，否则负号也会变方块。


In [ ]:
import matplotlib.pyplot as plt

# 直接贴这段就能让中文正常
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(8, 5))
plt.plot([1, 2, 3], [4, 5, 6])
plt.title('中文标题示例')
plt.xlabel('X轴标签')
plt.ylabel('Y轴标签')
plt.show()   # 输出: 中文标题、轴标签都正常


---
## 四、散点图核心参数（重点）

散点图比折线图多了一个本领：**用点的大小和颜色再塞进维度**。

| 参数 | 作用 | 示例 |
|---|---|---|
| `s` | 点的大小 | 标量（统一大小）或数组（每个点不同） |
| `c` | 点的颜色 | 颜色名/`#十六进制`，或数组（配合 `cmap` 映射） |
| `marker` | 点形状 | `'o'` 圆、`'s'` 方、`'^'` 三角 |
| `alpha` | 透明度 0~1 | `0.5` 半透明，点重叠时好看 |
| `cmap` | 颜色映射表 | `'viridis'`、`'coolwarm'`，配合 `c` 的数组用 |
| `edgecolors` | 点边框色 | `'white'` 让点有白边，更清晰 |


下面画一个"升级版"散点图：点的大小表示人数、颜色用 `cmap` 映射成绩高低，并加一条**回归线**看整体趋势。

> **原稿问题**：原稿用 `random.uniform` 随机生成"时长"和"成绩"，二者**毫无关系**，画出来的回归线是假趋势，容易误导。这里改用 numpy 生成**有正相关的数据**，回归线才有意义。另外原稿 `s=100,# 设置点的大小` 把注释写在参数逗号后面，语法上很别扭，下面改到独立一行。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(0)
hour = np.random.uniform(1, 5, 50)
score = 30 + 15 * hour + np.random.normal(0, 8, 50)

plt.figure(figsize=(10, 6))

# 回归线：一次拟合 y = a*x + b，polyfit 返回 [a, b]
a, b = np.polyfit(hour, score, 1)
x_line = np.linspace(hour.min(), hour.max(), 100)
plt.plot(x_line, a * x_line + b, color="red", linewidth=2, label="回归线")

# 散点：s 控制点大小，alpha 半透明，cmap 按成绩上色
plt.scatter(hour,
            score,
            label="成绩",
            marker="o",
            s=100,             # 点的大小
            c=score,           # 用成绩当颜色值
            cmap="coolwarm",   # 颜色映射表
            alpha=0.8,
            edgecolors="white")

plt.colorbar(label="成绩")     # 输出旁边一个颜色标尺
plt.title("学习时间与成绩的关系（含回归线）", fontsize=18)
plt.xlabel("学习时间（小时）")
plt.ylabel("成绩")
plt.legend()
plt.show()


---
## 五、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 只设一个中文字体 | 换台电脑就方块 | 给字体列表兜底 |
| 散点图当折线图用 | 点连成线、误导趋势 | 相关关系用 `scatter`，别用 `plot` |
| 随机数据硬画回归线 | 假趋势 | 先确认两变量真的相关 |
| 点太密看不清 | 一团黑 | 用 `alpha` 半透明、`s` 调小 |
| 注释写在参数逗号后 | 代码难读 | 注释单独成行 |

---

## 六、动手练习

**练习 1**：用 numpy 生成 `x = np.random.uniform(0, 10, 100)`、`y = 2*x + 噪声`，画散点图，点用蓝色、半透明。

**练习 2**：给练习 1 加一条回归线，验证斜率接近 2。

**练习 3**：用 `c` 参数让点的颜色随时间变化（比如 `c=np.arange(100)`），并加 `colorbar`。

**练习 4**：把点大小设成一个数组（比如 `s = 20 + 5*np.abs(y)`），观察大点小点的含义。

---

**答案思路（先自己做）**

```python
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(1)
x = np.random.uniform(0, 10, 100)
y = 2 * x + np.random.normal(0, 3, 100)

plt.figure()
plt.scatter(x, y, c=np.arange(100), cmap="viridis",
            s=20 + 5*np.abs(y), alpha=0.7, edgecolors="white")
plt.colorbar(label="序号")

a, b = np.polyfit(x, y, 1)
xs = np.linspace(0, 10, 50)
plt.plot(xs, a*xs + b, color="red", label=f"y={a:.2f}x+{b:.2f}")
plt.legend(); plt.show()
```
